# Chronic Kidney Disease Prediction 

In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score,classification_report,confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [36]:
data=pd.read_csv("CKD.csv")
data.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


In [37]:
data.shape

(399, 25)

In [38]:
data.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

In [39]:
data["classification"].value_counts()

classification
yes    249
no     150
Name: count, dtype: int64

In [40]:
data=pd.get_dummies(data,columns=['sg','rbc', 'pc', 'pcc', 'ba','htn', 'dm', 'cad','appet', 'pe', 'ane'],dtype=int)

In [41]:
data.head()


,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,dm_no,dm_yes,cad_no,cad_yes,appet_poor,appet_yes,pe_poor,pe_yes,ane_no,ane_yes
0,2.0,76.459948,3.0,0.0,148.112676,57.482105,3.077356,137.528754,4.627244,12.518156,...,1,0,1,0,0,1,0,1,1,0
1,3.0,76.459948,2.0,0.0,148.112676,22.000000,0.700000,137.528754,4.627244,10.700000,...,1,0,1,0,0,1,1,0,1,0
2,4.0,76.459948,1.0,0.0,99.000000,23.000000,0.600000,138.000000,4.400000,12.000000,...,1,0,1,0,0,1,1,0,1,0
3,5.0,76.459948,1.0,0.0,148.112676,16.000000,0.700000,138.000000,3.200000,8.100000,...,1,0,1,0,0,1,1,0,0,1
4,5.0,50.000000,0.0,0.0,148.112676,25.000000,0.600000,137.528754,4.627244,11.800000,...,1,0,1,0,0,1,1,0,1,0


In [42]:
X=data.drop(columns=["classification"])
y=data["classification"]

In [43]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [44]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

## Logistic Regression

In [47]:
param_grid={"solver":["lbfgs","liblinear","newton-cg","newton-cholesky","sag","saga"],"penalty":["l2"]}
lr_grid=GridSearchCV(LogisticRegression(),param_grid,refit=True,verbose=1,n_jobs=-1,cv=5,scoring="f1_weighted")
lr_grid.fit(X_train,y_train)
y_pred=lr_grid.predict(X_test)

from sklearn.metrics import classification_report,confusion_matrix

report=classification_report(y_test,y_pred)
print("\nClassification Report:")
print(report)
cm=confusion_matrix(y_test,y_pred)
print("\nConfusion Matirx:")
print(cm)

y_prob=lr_grid.predict_proba(X_test)[:,1]

print("\nAUC:",roc_auc_score(y_test,y_prob))

Fitting 5 folds for each of 6 candidates, totalling 30 fits

Classification Report:
              precision    recall  f1-score   support

          no       0.97      1.00      0.98        30
         yes       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80


Confusion Matirx:
[[30  0]
 [ 1 49]]

AUC: 1.0


## SVC

In [48]:
param_grid={"kernel":["linear", "poly", "rbf", "sigmoid"],"gamma":["scale","auto"],"C":[10,100,1000,2000,3000]}
svc_grid=GridSearchCV(SVC(probability=True),param_grid,refit=True,verbose=1,n_jobs=-1,cv=5,scoring="f1_weighted")
svc_grid.fit(X_train,y_train)
y_pred=svc_grid.predict(X_test)

report=classification_report(y_test,y_pred)
print("\nClassification Report:")
print(report)
cm=confusion_matrix(y_test,y_pred)
print("\nConfusion Matirx:")
print(cm)

y_prob=svc_grid.predict_proba(X_test)[:,1]

print("\nAUC:",roc_auc_score(y_test,y_prob))

Fitting 5 folds for each of 40 candidates, totalling 200 fits

Classification Report:
              precision    recall  f1-score   support

          no       0.94      1.00      0.97        30
         yes       1.00      0.96      0.98        50

    accuracy                           0.97        80
   macro avg       0.97      0.98      0.97        80
weighted avg       0.98      0.97      0.98        80


Confusion Matirx:
[[30  0]
 [ 2 48]]

AUC: 1.0


In [ ]:
## DT

In [49]:
param_grid={"criterion":['gini','entropy', 'log_loss'],"splitter":['best', 'random']}
dt_grid=GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,cv=5,scoring="f1_weighted")
dt_grid.fit(X_train,y_train)
y_pred=dt_grid.predict(X_test)

report=classification_report(y_test,y_pred)
print("\nClassification Report:")
print(report)
cm=confusion_matrix(y_test,y_pred)
print("\nConfusion Matirx:")
print(cm)

y_prob=dt_grid.predict_proba(X_test)[:,1]

print("\nAUC:",roc_auc_score(y_test,y_prob))

Fitting 5 folds for each of 6 candidates, totalling 30 fits

Classification Report:
              precision    recall  f1-score   support

          no       0.97      1.00      0.98        30
         yes       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80


Confusion Matirx:
[[30  0]
 [ 1 49]]

AUC: 0.99


## RF

In [50]:
param_grid={"criterion":['gini','entropy', 'log_loss'],"n_estimators":[10,100]}
rf_grid=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,cv=5,scoring="f1_weighted")
rf_grid.fit(X_train,y_train)
y_pred=rf_grid.predict(X_test)

report=classification_report(y_test,y_pred)
print("\nClassification Report:")
print(report)
cm=confusion_matrix(y_test,y_pred)
print("\nConfusion Matirx:")
print(cm)

y_prob=rf_grid.predict_proba(X_test)[:,1]

print("\nAUC:",roc_auc_score(y_test,y_prob))

Fitting 5 folds for each of 6 candidates, totalling 30 fits

Classification Report:
              precision    recall  f1-score   support

          no       0.97      1.00      0.98        30
         yes       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80


Confusion Matirx:
[[30  0]
 [ 1 49]]

AUC: 1.0


In [51]:
print("LR:", lr_grid.best_score_)
print("SVC:", svc_grid.best_score_)
print("DT:", dt_grid.best_score_)
print("RF:", rf_grid.best_score_)

LR: 0.9843613949771919
SVC: 0.9844319810689612
DT: 0.9906613278222682
RF: 0.9873222599031534


In [52]:
print(lr_grid.best_params_)
print(svc_grid.best_params_)
print(dt_grid.best_params_)
print(rf_grid.best_params_)

{'penalty': 'l2', 'solver': 'lbfgs'}
{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}
{'criterion': 'log_loss', 'splitter': 'random'}
{'criterion': 'entropy', 'n_estimators': 100}


# Observation

### The Random Forest model performed best with a 98.73% weighted F1-score in cross-validation, followed closely by Decision Tree (99.07%), SVC (98.44%), and Logistic Regression (98.44%); all four models showed strong cross-validation performance, with further comparison needed on unseen test data.